# Clase 12 · Notebook 01
# Optimizadores e inicialización de pesos

**Introducción al Deep Learning — Módulo 2, Unidad 1: Redes de neuronas (Parte II)**

Dos decisiones que cambian mucho cómo entrena una red:

1. El **optimizador** (SGD vs Adam).
2. La **inicialización** de los pesos (aleatoria simple vs He).

Comparamos ambas cosas dibujando las **curvas de pérdida**.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(42); np.random.seed(42)

data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data); y = data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Datos listos:", Xtr.shape)

## 2. Comparar optimizadores: SGD vs Adam

Construimos la misma red dos veces, cambiando solo el optimizador, y comparamos sus curvas de pérdida.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def crear(optimizador):
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(X.shape[1],)),
                    Dense(32, activation="relu"),
                    Dense(32, activation="relu"),
                    Dense(1, activation="sigmoid")])
    m.compile(optimizer=optimizador, loss="binary_crossentropy", metrics=["accuracy"])
    return m

h_sgd  = crear(tf.keras.optimizers.SGD(0.01)).fit(Xtr, ytr, epochs=60, batch_size=16,
                                                  validation_split=0.1, verbose=0)
h_adam = crear(tf.keras.optimizers.Adam(0.01)).fit(Xtr, ytr, epochs=60, batch_size=16,
                                                  validation_split=0.1, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(h_sgd.history["loss"],  label="SGD",  color="#4355FF")
plt.plot(h_adam.history["loss"], label="Adam", color="#FF647E")
plt.title("Pérdida de entrenamiento: SGD vs Adam")
plt.xlabel("Época"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Loss final SGD : %.4f" % h_sgd.history["loss"][-1])
print("Loss final Adam: %.4f" % h_adam.history["loss"][-1])

Normalmente **Adam baja la pérdida más rápido** que SGD con la misma tasa de aprendizaje, porque adapta
el paso para cada parámetro.

## 3. Comparar inicialización: aleatoria simple vs He

La inicialización **He** está pensada para ReLU y ayuda a que el gradiente fluya bien en redes profundas.
Comparamos contra una inicialización aleatoria de varianza fija (`RandomNormal`).

In [ ]:
from tensorflow.keras.initializers import HeNormal, RandomNormal

def crear_init(inicializador):
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(X.shape[1],))] +
                   [Dense(64, activation="relu", kernel_initializer=inicializador) for _ in range(4)] +
                   [Dense(1, activation="sigmoid")])
    m.compile(optimizer=tf.keras.optimizers.Adam(0.005), loss="binary_crossentropy", metrics=["accuracy"])
    return m

h_rand = crear_init(RandomNormal(stddev=0.05)).fit(Xtr, ytr, epochs=60, batch_size=16, verbose=0)
h_he   = crear_init(HeNormal()).fit(Xtr, ytr, epochs=60, batch_size=16, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(h_rand.history["loss"], label="Aleatoria (stddev fija)", color="#4355FF")
plt.plot(h_he.history["loss"],   label="He", color="#FF647E")
plt.title("Pérdida según la inicialización (red profunda, ReLU)")
plt.xlabel("Época"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Loss final aleatoria: %.4f" % h_rand.history["loss"][-1])
print("Loss final He       : %.4f" % h_he.history["loss"][-1])

En una red profunda con ReLU, **He suele arrancar y converger mejor** que una inicialización aleatoria
de varianza arbitraria, que puede hacer que el aprendizaje sea lento al principio.

## Resumen

- El **optimizador** marca la velocidad y estabilidad del entrenamiento: **Adam** es un gran punto de partida.
- La **tasa de aprendizaje** es el hiperparámetro más influyente.
- La **inicialización** adecuada (**He** para ReLU, **Glorot** para sigmoide/tanh) evita gradientes que se
  desvanecen o explotan en redes profundas.

➡️ En el **Notebook 02** atacaremos el **sobreajuste** con dropout, batch norm y early stopping.